In [10]:
import json
import os
import sys
from pathlib import Path
from typing import Literal

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import Runnable, RunnableLambda, RunnableParallel, RunnablePassthrough
from langchain_openai import ChatOpenAI
from pydantic import BaseModel, Field


# 프로젝트 루트 경로를 찾아 src 폴더를 sys.path에 추가한다.
def find_project_root() -> Path:
    """현재 작업 디렉터리에서 프로젝트 루트를 탐색한다."""
    curr = Path.cwd()
    for parent in [curr] + list(curr.parents):
        if (parent / "pyproject.toml").exists():
            return parent
    return curr


PROJECT_ROOT = find_project_root()
USER_DATA_PATH = PROJECT_ROOT / "notebook/team02/02_Layer4/user_data.json"
src_dir = str(PROJECT_ROOT / "src")
if src_dir not in sys.path:
    sys.path.insert(0, src_dir)

if os.getenv("LANGSMITH_API_KEY", "").strip():
    os.environ.setdefault("LANGSMITH_TRACING", "true")
else:
    os.environ["LANGSMITH_TRACING"] = "false"

In [11]:
class SourcePaths(BaseModel):
    """사용자 소비 JSON의 원천 데이터 경로를 표현한다."""

    past_csv: str
    today_csv: str


class OutlierThresholds(BaseModel):
    """이상 지출 판단에 사용한 사분위수 기준값을 표현한다."""

    q1: float
    q3: float
    iqr: float
    lower_bound: float
    upper_bound: float


class CategoryRatioChange(BaseModel):
    """카테고리별 평소 대비 당일 소비 비중 변화를 표현한다."""

    category: str
    usual_ratio_percent: float
    today_ratio_percent: float
    diff_point: float


class StableMetrics(BaseModel):
    """클리핑 데이터 기반 안정 소비 지표를 표현한다."""

    past_daily_stable_average: float
    today_total: int
    increase_rate_percent: float
    category_ratio_changes: list[CategoryRatioChange]


class HighSpendingItem(BaseModel):
    """원본 데이터에서 탐지한 특이 고액 지출 항목을 표현한다."""

    used_at: str
    description: str
    amount: int
    category: str


class AnomalyDetection(BaseModel):
    """원본 데이터 기반 소비 규모와 이상 지출 탐지 결과를 표현한다."""

    past_daily_original_average: float
    spike_ratio: float
    is_spike: bool
    high_spending_threshold: float
    high_spending_items: list[HighSpendingItem]


class PreviousDayComparison(BaseModel):
    """전날 대비 소비 비교 결과를 표현한다."""

    yesterday_date: str
    yesterday_total: int
    today_total: int
    amount_diff: int
    amount_diff_rate_percent: float
    yesterday_count: int
    today_count: int
    count_diff: int
    yesterday_main_category: str
    today_main_category: str


class TimeSlotComparison(BaseModel):
    """시간대별 당일 소비와 평소 소비 차이를 표현한다."""

    time_slot: str
    today_amount: int
    usual_average_amount: float
    diff_amount: float


class TimeSlotAnalysis(BaseModel):
    """시간대별 소비 피크와 세부 비교 결과를 표현한다."""

    peak_slot: str
    time_slots: list[TimeSlotComparison]


class UserSpendingData(BaseModel):
    """user_data.json 전체 구조를 검증 가능한 입력 모델로 표현한다."""

    member_id: int
    analysis_date: str
    source_paths: SourcePaths
    outlier_thresholds: OutlierThresholds
    stable_metrics: StableMetrics
    anomaly_detection: AnomalyDetection
    previous_day_comparison: PreviousDayComparison
    time_slot_analysis: TimeSlotAnalysis


MetricValue = int | float | str | bool
CategoryDirection = Literal["increase", "decrease", "flat"]


class SpendingMetric(BaseModel):
    """JSON 데이터에서 바로 추출한 단일 소비 지표를 표현한다."""

    name: str
    value: MetricValue
    unit: str
    source_json_path: str
    description: str


class CategoryShiftIndicator(BaseModel):
    """카테고리 비중 변화 지표와 원본 JSON 경로를 함께 표현한다."""

    category: str
    usual_ratio_percent: float
    today_ratio_percent: float
    diff_point: float
    direction: CategoryDirection
    source_json_path: str


class MainCategoryShift(BaseModel):
    """전날 대비 주 소비 카테고리 변화 지표를 표현한다."""

    previous_category: str
    current_category: str
    source_json_path: str


class SpendingIndicatorPayload(BaseModel):
    """분석 체인에 직접 전달할 JSON 기반 소비 지표 묶음을 표현한다."""

    member_id: int
    analysis_date: str
    metrics: list[SpendingMetric]
    category_ratio_changes: list[CategoryShiftIndicator]
    largest_category_increase: CategoryShiftIndicator | None
    largest_category_decrease: CategoryShiftIndicator | None
    high_spending_items: list[HighSpendingItem]
    main_category_shift: MainCategoryShift
    time_slot_diffs: list[TimeSlotComparison]


def load_user_spending_data(json_path: Path) -> UserSpendingData:
    """사용자 소비 JSON 파일을 읽어 검증된 입력 모델로 변환한다."""
    with json_path.open(encoding="utf-8") as file:
        raw_payload: object = json.load(file)
    return UserSpendingData.model_validate(raw_payload)


def get_category_direction(diff_point: float) -> CategoryDirection:
    """카테고리 비중 차이를 증가, 감소, 변화 없음으로 분류한다."""
    if diff_point > 0:
        return "increase"
    if diff_point < 0:
        return "decrease"
    return "flat"


def make_spending_metric(
    name: str,
    value: MetricValue,
    unit: str,
    source_json_path: str,
    description: str,
) -> SpendingMetric:
    """소비 지표 이름, 값, 단위, JSON 경로를 하나의 모델로 묶는다."""
    return SpendingMetric(
        name=name,
        value=value,
        unit=unit,
        source_json_path=source_json_path,
        description=description,
    )


def build_category_shift_indicators(
    changes: list[CategoryRatioChange],
) -> list[CategoryShiftIndicator]:
    """카테고리 비중 변화 원본 배열을 JSON 경로가 포함된 지표 목록으로 변환한다."""
    return [
        CategoryShiftIndicator(
            category=change.category,
            usual_ratio_percent=change.usual_ratio_percent,
            today_ratio_percent=change.today_ratio_percent,
            diff_point=change.diff_point,
            direction=get_category_direction(change.diff_point),
            source_json_path=f"stable_metrics.category_ratio_changes[{index}]",
        )
        for index, change in enumerate(changes)
    ]


def get_largest_category_increase(
    changes: list[CategoryShiftIndicator],
) -> CategoryShiftIndicator | None:
    """비중이 가장 크게 증가한 카테고리 지표를 찾는다."""
    increased_changes = [change for change in changes if change.direction == "increase"]
    if not increased_changes:
        return None
    return max(increased_changes, key=lambda change: change.diff_point)


def get_largest_category_decrease(
    changes: list[CategoryShiftIndicator],
) -> CategoryShiftIndicator | None:
    """비중이 가장 크게 감소한 카테고리 지표를 찾는다."""
    decreased_changes = [change for change in changes if change.direction == "decrease"]
    if not decreased_changes:
        return None
    return min(decreased_changes, key=lambda change: change.diff_point)


def get_largest_high_spending_item(items: list[HighSpendingItem]) -> HighSpendingItem | None:
    """특이 지출 항목 중 금액이 가장 큰 항목을 찾는다."""
    if not items:
        return None
    return max(items, key=lambda item: item.amount)


def build_core_metrics(user_data: UserSpendingData) -> list[SpendingMetric]:
    """분석에 자주 쓰는 핵심 소비 지표를 원본 JSON에서 직접 추출한다."""
    stable_metrics = user_data.stable_metrics
    anomaly = user_data.anomaly_detection
    previous_day = user_data.previous_day_comparison
    high_spending_item = get_largest_high_spending_item(anomaly.high_spending_items)

    metrics = [
        make_spending_metric(
            "과거 안정 일평균",
            stable_metrics.past_daily_stable_average,
            "KRW",
            "stable_metrics.past_daily_stable_average",
            "클리핑 데이터 기준 과거 일평균 소비 금액",
        ),
        make_spending_metric(
            "오늘 총 지출액",
            stable_metrics.today_total,
            "KRW",
            "stable_metrics.today_total",
            "분석 기준일의 총 소비 금액",
        ),
        make_spending_metric(
            "평소 대비 지출 증가율",
            stable_metrics.increase_rate_percent,
            "percent",
            "stable_metrics.increase_rate_percent",
            "과거 안정 일평균 대비 오늘 총 지출 증가율",
        ),
        make_spending_metric(
            "원본 일평균 대비 소비 배율",
            anomaly.spike_ratio,
            "ratio",
            "anomaly_detection.spike_ratio",
            "원본 데이터 과거 일평균 대비 오늘 지출 배율",
        ),
        make_spending_metric(
            "소비 급증 여부",
            anomaly.is_spike,
            "boolean",
            "anomaly_detection.is_spike",
            "오늘 지출이 이상 소비 기준을 넘었는지 여부",
        ),
        make_spending_metric(
            "고액 지출 기준값",
            anomaly.high_spending_threshold,
            "KRW",
            "anomaly_detection.high_spending_threshold",
            "건별 고액 지출 판단에 사용한 상한 기준값",
        ),
        make_spending_metric(
            "고액 지출 건수",
            len(anomaly.high_spending_items),
            "count",
            "anomaly_detection.high_spending_items",
            "고액 지출 기준값을 초과한 결제 항목 수",
        ),
        make_spending_metric(
            "전날 대비 지출 증감액",
            previous_day.amount_diff,
            "KRW",
            "previous_day_comparison.amount_diff",
            "전날 총 지출과 오늘 총 지출의 차이",
        ),
        make_spending_metric(
            "전날 대비 지출 증감률",
            previous_day.amount_diff_rate_percent,
            "percent",
            "previous_day_comparison.amount_diff_rate_percent",
            "전날 총 지출 대비 오늘 지출 증감률",
        ),
        make_spending_metric(
            "전날 대비 결제 건수 증감",
            previous_day.count_diff,
            "count",
            "previous_day_comparison.count_diff",
            "전날 결제 건수와 오늘 결제 건수의 차이",
        ),
        make_spending_metric(
            "오늘 소비 피크 시간대",
            user_data.time_slot_analysis.peak_slot,
            "time_slot",
            "time_slot_analysis.peak_slot",
            "분석 기준일 지출액이 가장 큰 시간대",
        ),
    ]

    if high_spending_item:
        metrics.append(
            make_spending_metric(
                "최대 고액 지출 금액",
                high_spending_item.amount,
                "KRW",
                "anomaly_detection.high_spending_items",
                f"고액 지출 항목 중 가장 큰 결제 금액: {high_spending_item.description}",
            )
        )

    return metrics


def extract_spending_indicators(user_data: UserSpendingData) -> SpendingIndicatorPayload:
    """user_data.json에서 분석 가능한 소비 지표를 마크다운 변환 없이 직접 추출한다."""
    category_changes = build_category_shift_indicators(
        user_data.stable_metrics.category_ratio_changes
    )
    previous_day = user_data.previous_day_comparison
    return SpendingIndicatorPayload(
        member_id=user_data.member_id,
        analysis_date=user_data.analysis_date,
        metrics=build_core_metrics(user_data),
        category_ratio_changes=category_changes,
        largest_category_increase=get_largest_category_increase(category_changes),
        largest_category_decrease=get_largest_category_decrease(category_changes),
        high_spending_items=user_data.anomaly_detection.high_spending_items,
        main_category_shift=MainCategoryShift(
            previous_category=previous_day.yesterday_main_category,
            current_category=previous_day.today_main_category,
            source_json_path="previous_day_comparison",
        ),
        time_slot_diffs=user_data.time_slot_analysis.time_slots,
    )


def make_analysis_input(user_data: UserSpendingData) -> dict[str, str]:
    """소비 분석 체인에 넣을 원본 JSON과 추출 지표 JSON 입력을 생성한다."""
    spending_indicators = extract_spending_indicators(user_data)
    return {
        "raw_json": user_data.model_dump_json(indent=2),
        "indicator_json": spending_indicators.model_dump_json(indent=2),
    }

In [12]:
user_spending_data = load_user_spending_data(USER_DATA_PATH)
spending_indicators = extract_spending_indicators(user_spending_data)
analysis_input = make_analysis_input(user_spending_data)
metric_by_name = {metric.name: metric for metric in spending_indicators.metrics}

assert user_spending_data.member_id == 1
assert user_spending_data.analysis_date == "2024-04-01"
assert metric_by_name["오늘 총 지출액"].value == 133044
assert metric_by_name["평소 대비 지출 증가율"].source_json_path == (
    "stable_metrics.increase_rate_percent"
)
assert metric_by_name["소비 급증 여부"].value is False
assert metric_by_name["고액 지출 건수"].value == 1
assert spending_indicators.largest_category_increase is not None
assert spending_indicators.largest_category_increase.category == "생활"
assert spending_indicators.largest_category_decrease is not None
assert spending_indicators.largest_category_decrease.category == "식비"
assert spending_indicators.high_spending_items[0].description == "SKT통신비"
assert spending_indicators.main_category_shift.previous_category == "식비"
assert spending_indicators.main_category_shift.current_category == "생활"
assert spending_indicators.time_slot_diffs[0].time_slot == "2.오전(06-11)"
assert "indicator_json" in analysis_input
assert "raw_json" in analysis_input
assert "report_text" not in analysis_input
assert "stable_metrics.today_total" in analysis_input["indicator_json"]
assert not analysis_input["indicator_json"].lstrip().startswith("#")

print(analysis_input["indicator_json"])

{
  "member_id": 1,
  "analysis_date": "2024-04-01",
  "metrics": [
    {
      "name": "과거 안정 일평균",
      "value": 51014.0549,
      "unit": "KRW",
      "source_json_path": "stable_metrics.past_daily_stable_average",
      "description": "클리핑 데이터 기준 과거 일평균 소비 금액"
    },
    {
      "name": "오늘 총 지출액",
      "value": 133044,
      "unit": "KRW",
      "source_json_path": "stable_metrics.today_total",
      "description": "분석 기준일의 총 소비 금액"
    },
    {
      "name": "평소 대비 지출 증가율",
      "value": 160.7987,
      "unit": "percent",
      "source_json_path": "stable_metrics.increase_rate_percent",
      "description": "과거 안정 일평균 대비 오늘 총 지출 증가율"
    },
    {
      "name": "원본 일평균 대비 소비 배율",
      "value": 1.2741,
      "unit": "ratio",
      "source_json_path": "anomaly_detection.spike_ratio",
      "description": "원본 데이터 과거 일평균 대비 오늘 지출 배율"
    },
    {
      "name": "소비 급증 여부",
      "value": false,
      "unit": "boolean",
      "source_json_path": "anomaly_detection.is_spike",
      "

In [13]:
FindingConfidence = Literal["low", "medium", "high"]


class EvidenceItem(BaseModel):
    """JSON 지표 분석에서 참조한 근거 값을 표현한다."""

    json_path: str = Field(description="근거가 나온 원본 또는 지표 JSON 경로")
    supporting_value: str = Field(description="근거가 된 JSON 값")
    reason: str = Field(description="이 근거를 선택한 이유")


class SpendingFinding(BaseModel):
    """소비 분석의 단일 탐지 결과를 표현한다."""

    subcategory: str = Field(description="세부 분석 분류")
    title: str = Field(description="한 줄 요약")
    detail: str = Field(description="근거를 포함한 구체적인 설명")
    confidence: FindingConfidence = Field(description="판단 신뢰도")
    evidences: list[EvidenceItem] = Field(description="판단에 사용한 JSON 근거 목록")


class PatternAnalysisResult(BaseModel):
    """소비 패턴 탐지 결과를 구조화한다."""

    repeated_consumption: list[SpendingFinding] = Field(default_factory=list)
    overspending_windows: list[SpendingFinding] = Field(default_factory=list)
    impulse_patterns: list[SpendingFinding] = Field(default_factory=list)
    contextual_patterns: list[SpendingFinding] = Field(default_factory=list)


class ProblemAnalysisResult(BaseModel):
    """문제 소비 식별 결과를 구조화한다."""

    money_leaks: list[SpendingFinding] = Field(default_factory=list)
    saving_blockers: list[SpendingFinding] = Field(default_factory=list)
    fixed_cost_issues: list[SpendingFinding] = Field(default_factory=list)
    variable_cost_issues: list[SpendingFinding] = Field(default_factory=list)
    short_term_problem_spending: list[SpendingFinding] = Field(default_factory=list)
    long_term_problem_spending: list[SpendingFinding] = Field(default_factory=list)


class CauseAnalysisResult(BaseModel):
    """소비 원인 해석 결과를 구조화한다."""

    habitual_causes: list[SpendingFinding] = Field(default_factory=list)
    reward_causes: list[SpendingFinding] = Field(default_factory=list)
    stress_causes: list[SpendingFinding] = Field(default_factory=list)
    convenience_causes: list[SpendingFinding] = Field(default_factory=list)
    small_accumulation_causes: list[SpendingFinding] = Field(default_factory=list)


class ActionMission(BaseModel):
    """행동 개선 포인트와 실행 미션을 표현한다."""

    action_type: str = Field(description="행동 포인트 유형")
    title: str = Field(description="실행 항목 제목")
    detail: str = Field(description="실행 방법 설명")
    target_json_path: str = Field(description="직접 연결되는 원본 또는 지표 JSON 경로")
    expected_effect: str = Field(description="기대 효과")
    urgency: Literal["immediate", "this_week", "this_month"] = Field(
        description="실행 우선순위 시점"
    )


class GroupCompetitionMetric(BaseModel):
    """그룹 경쟁에 반영할 행동 지표를 표현한다."""

    metric_name: str = Field(description="행동 지표 이름")
    definition: str = Field(description="행동 지표 계산 규칙")
    target_value: str = Field(description="권장 목표값")
    reason: str = Field(description="이 지표를 추천하는 이유")


class ActionAnalysisResult(BaseModel):
    """행동 개선 포인트 도출 결과를 구조화한다."""

    immediate_cuts: list[ActionMission] = Field(default_factory=list)
    substitution_opportunities: list[ActionMission] = Field(default_factory=list)
    budget_control_areas: list[ActionMission] = Field(default_factory=list)
    next_week_missions: list[ActionMission] = Field(default_factory=list)
    group_competition_metrics: list[GroupCompetitionMetric] = Field(default_factory=list)


def build_pattern_prompt() -> ChatPromptTemplate:
    """JSON 소비 지표 기반 소비 패턴 탐지용 프롬프트를 생성한다."""
    return ChatPromptTemplate.from_messages(
        [
            (
                "system",
                "당신은 카드 소비 JSON 데이터를 해석하는 금융 코치다. 제공된 원본 JSON과 추출 지표 JSON만 사용해 판단하고, "
                "마크다운 소비 보고서로 재구성하지 마라. 근거가 부족하면 confidence를 낮게 설정하라. "
                "모든 응답은 한국어로 작성하고 evidences.json_path에는 실제 JSON 경로를 적어라.",
            ),
            (
                "human",
                "아래 JSON 데이터를 바탕으로 소비 패턴을 탐지하라.\n"
                "반드시 반복 소비, 과소비 구간, 충동소비 의심 패턴, 시간대/상황별 소비 패턴을 각각 채워라.\n\n"
                "원본 JSON:\n{raw_json}\n\n"
                "추출 지표 JSON:\n{indicator_json}",
            ),
        ]
    )


def build_problem_prompt() -> ChatPromptTemplate:
    """JSON 소비 지표 기반 문제 소비 식별용 프롬프트를 생성한다."""
    return ChatPromptTemplate.from_messages(
        [
            (
                "system",
                "당신은 사용자의 절약 실패 원인을 JSON 지표로 분리해서 설명하는 소비 분석가다. "
                "제공된 JSON 값만 사용하고 추측성 서술은 최소화하라. "
                "고정비와 변동비 문제를 분리하고, 단기 문제와 장기 문제를 구분하라.",
            ),
            (
                "human",
                "아래 JSON 데이터를 바탕으로 문제 소비를 식별하라.\n"
                "새는 돈 포인트, 절약 방해 요소, 고정비 문제, 변동비 문제, 단기 문제 소비, 장기 문제 소비를 채워라.\n\n"
                "원본 JSON:\n{raw_json}\n\n"
                "추출 지표 JSON:\n{indicator_json}",
            ),
        ]
    )


def build_cause_prompt() -> ChatPromptTemplate:
    """JSON 소비 지표 기반 소비 원인 해석용 프롬프트를 생성한다."""
    return ChatPromptTemplate.from_messages(
        [
            (
                "system",
                "당신은 이미 식별된 패턴과 문제 소비를 바탕으로 행동 원인을 해석하는 분석가다. "
                "원본 JSON과 추출 지표 JSON의 수치 근거를 우선 사용하라. "
                "습관성, 보상성, 스트레스성, 편의성 기반, 소액 누적형 소비를 구분해 설명하라.",
            ),
            (
                "human",
                "아래 JSON 데이터와 선행 분석 결과를 바탕으로 소비 원인을 해석하라.\n\n"
                "원본 JSON:\n{raw_json}\n\n"
                "추출 지표 JSON:\n{indicator_json}\n\n"
                "패턴 탐지 결과:\n{pattern_text}\n\n"
                "문제 소비 결과:\n{problem_text}",
            ),
        ]
    )


def build_action_prompt() -> ChatPromptTemplate:
    """JSON 소비 지표 기반 행동 개선 포인트 도출용 프롬프트를 생성한다."""
    return ChatPromptTemplate.from_messages(
        [
            (
                "system",
                "당신은 소비 코칭 액션 플랜을 만드는 코치다. 즉시 줄일 수 있는 소비, 대체 가능한 소비, "
                "예산 통제가 필요한 영역, 다음 주 행동 미션, 그룹 경쟁 지표를 구체적으로 제안하라. "
                "행동 항목은 짧고 실행 가능해야 하며 target_json_path에는 연결되는 JSON 경로를 적어라.",
            ),
            (
                "human",
                "아래 JSON 데이터와 선행 분석을 바탕으로 행동 개선 포인트를 도출하라.\n\n"
                "원본 JSON:\n{raw_json}\n\n"
                "추출 지표 JSON:\n{indicator_json}\n\n"
                "패턴 탐지 결과:\n{pattern_text}\n\n"
                "문제 소비 결과:\n{problem_text}\n\n"
                "소비 원인 결과:\n{cause_text}",
            ),
        ]
    )

In [14]:
pattern_prompt = build_pattern_prompt()
problem_prompt = build_problem_prompt()
cause_prompt = build_cause_prompt()
action_prompt = build_action_prompt()

assert set(pattern_prompt.input_variables) == {"raw_json", "indicator_json"}
assert set(problem_prompt.input_variables) == {"raw_json", "indicator_json"}
assert set(cause_prompt.input_variables) == {
    "raw_json",
    "indicator_json",
    "pattern_text",
    "problem_text",
}
assert set(action_prompt.input_variables) == {
    "raw_json",
    "indicator_json",
    "pattern_text",
    "problem_text",
    "cause_text",
}

rendered_pattern = pattern_prompt.invoke(analysis_input)
rendered_pattern_content = str(rendered_pattern.messages[-1].content)
assert "stable_metrics.today_total" in rendered_pattern_content
assert "SKT통신비" in rendered_pattern_content
assert "# 멤버" not in rendered_pattern_content
assert "마크다운 소비 보고서" not in rendered_pattern_content

rendered_cause = cause_prompt.invoke(
    {
        "raw_json": analysis_input["raw_json"],
        "indicator_json": analysis_input["indicator_json"],
        "pattern_text": "sample pattern",
        "problem_text": "sample problem",
    }
)
rendered_cause_content = str(rendered_cause.messages[-1].content)
assert "sample pattern" in rendered_cause_content
assert "sample problem" in rendered_cause_content
assert "추출 지표 JSON" in rendered_cause_content

In [15]:
def prepare_cause_payload(payload: dict[str, object]) -> dict[str, str]:
    """원인 해석 체인에 필요한 JSON 입력 페이로드를 생성한다."""
    pattern_result = payload["pattern_result"]
    problem_result = payload["problem_result"]
    assert isinstance(pattern_result, PatternAnalysisResult)
    assert isinstance(problem_result, ProblemAnalysisResult)
    return {
        "raw_json": str(payload["raw_json"]),
        "indicator_json": str(payload["indicator_json"]),
        "pattern_text": pattern_result.model_dump_json(indent=2),
        "problem_text": problem_result.model_dump_json(indent=2),
    }


def prepare_action_payload(payload: dict[str, object]) -> dict[str, str]:
    """행동 개선 체인에 필요한 JSON 입력 페이로드를 생성한다."""
    pattern_result = payload["pattern_result"]
    problem_result = payload["problem_result"]
    cause_result = payload["cause_result"]
    assert isinstance(pattern_result, PatternAnalysisResult)
    assert isinstance(problem_result, ProblemAnalysisResult)
    assert isinstance(cause_result, CauseAnalysisResult)
    return {
        "raw_json": str(payload["raw_json"]),
        "indicator_json": str(payload["indicator_json"]),
        "pattern_text": pattern_result.model_dump_json(indent=2),
        "problem_text": problem_result.model_dump_json(indent=2),
        "cause_text": cause_result.model_dump_json(indent=2),
    }


def create_notebook_llm(model_name: str = "gpt-4o-mini") -> ChatOpenAI:
    """노트북 실행용 ChatOpenAI 모델을 생성한다."""
    return ChatOpenAI(model=model_name, temperature=0)


def build_pattern_chain(llm: ChatOpenAI) -> Runnable[dict[str, str], PatternAnalysisResult]:
    """소비 패턴 탐지 체인을 생성한다."""
    return build_pattern_prompt() | llm.with_structured_output(PatternAnalysisResult)


def build_problem_chain(llm: ChatOpenAI) -> Runnable[dict[str, str], ProblemAnalysisResult]:
    """문제 소비 식별 체인을 생성한다."""
    return build_problem_prompt() | llm.with_structured_output(ProblemAnalysisResult)


def build_cause_chain(llm: ChatOpenAI) -> Runnable[dict[str, str], CauseAnalysisResult]:
    """소비 원인 해석 체인을 생성한다."""
    return build_cause_prompt() | llm.with_structured_output(CauseAnalysisResult)


def build_action_chain(llm: ChatOpenAI) -> Runnable[dict[str, str], ActionAnalysisResult]:
    """행동 개선 포인트 도출 체인을 생성한다."""
    return build_action_prompt() | llm.with_structured_output(ActionAnalysisResult)


def build_spending_analysis_chain(llm: ChatOpenAI) -> Runnable[dict[str, str], dict[str, object]]:
    """JSON 지표 기반 소비 패턴, 문제 소비, 원인, 행동 포인트 체인을 생성한다."""
    diagnosis_chain = RunnableParallel(
        raw_json=RunnableLambda(lambda payload: str(payload["raw_json"])),
        indicator_json=RunnableLambda(lambda payload: str(payload["indicator_json"])),
        pattern_result=build_pattern_chain(llm),
        problem_result=build_problem_chain(llm),
    )
    return (
        diagnosis_chain
        | RunnablePassthrough.assign(
            cause_result=RunnableLambda(prepare_cause_payload) | build_cause_chain(llm)
        )
        | RunnablePassthrough.assign(
            action_result=RunnableLambda(prepare_action_payload) | build_action_chain(llm)
        )
    )

In [16]:
pattern_prompt_preview = build_pattern_prompt().invoke(analysis_input).messages[-1].content
print(str(pattern_prompt_preview)[:1200])

아래 JSON 데이터를 바탕으로 소비 패턴을 탐지하라.
반드시 반복 소비, 과소비 구간, 충동소비 의심 패턴, 시간대/상황별 소비 패턴을 각각 채워라.

원본 JSON:
{
  "member_id": 1,
  "analysis_date": "2024-04-01",
  "source_paths": {
    "past_csv": "../../../data/raw/csv/transactions_v1.csv",
    "today_csv": "data_input_month.csv"
  },
  "outlier_thresholds": {
    "q1": 5242.0,
    "q3": 12247.0,
    "iqr": 7005.0,
    "lower_bound": 0.0,
    "upper_bound": 22754.5
  },
  "stable_metrics": {
    "past_daily_stable_average": 51014.0549,
    "today_total": 133044,
    "increase_rate_percent": 160.7987,
    "category_ratio_changes": [
      {
        "category": "생활",
        "usual_ratio_percent": 1.4705,
        "today_ratio_percent": 72.833,
        "diff_point": 71.3626
      },
      {
        "category": "교통",
        "usual_ratio_percent": 4.7418,
        "today_ratio_percent": 9.6885,
        "diff_point": 4.9468
      },
      {
        "category": "의료",
        "usual_ratio_percent": 8.0934,
        "today_ratio_percent": 10.0929,
        "

In [17]:
openai_api_key = os.getenv("OPENAI_API_KEY", "").strip()

if openai_api_key:
    spending_analysis_chain = build_spending_analysis_chain(create_notebook_llm())
    analysis_result = spending_analysis_chain.invoke(analysis_input)
    print(analysis_result)
else:
    analysis_result = None
    print(
        "OPENAI_API_KEY가 없어 체인 실행은 건너뜁니다. "
        "JSON 로드, 지표 추출, 프롬프트와 체인 정의 셀까지만 검증했습니다."
    )

{'raw_json': '{\n  "member_id": 1,\n  "analysis_date": "2024-04-01",\n  "source_paths": {\n    "past_csv": "../../../data/raw/csv/transactions_v1.csv",\n    "today_csv": "data_input_month.csv"\n  },\n  "outlier_thresholds": {\n    "q1": 5242.0,\n    "q3": 12247.0,\n    "iqr": 7005.0,\n    "lower_bound": 0.0,\n    "upper_bound": 22754.5\n  },\n  "stable_metrics": {\n    "past_daily_stable_average": 51014.0549,\n    "today_total": 133044,\n    "increase_rate_percent": 160.7987,\n    "category_ratio_changes": [\n      {\n        "category": "생활",\n        "usual_ratio_percent": 1.4705,\n        "today_ratio_percent": 72.833,\n        "diff_point": 71.3626\n      },\n      {\n        "category": "교통",\n        "usual_ratio_percent": 4.7418,\n        "today_ratio_percent": 9.6885,\n        "diff_point": 4.9468\n      },\n      {\n        "category": "의료",\n        "usual_ratio_percent": 8.0934,\n        "today_ratio_percent": 10.0929,\n        "diff_point": 1.9995\n      },\n      {\n      

In [18]:
def print_pretty_json_analysis(result: object | None) -> None:
    """체인 결과를 입력 JSON 지표와 함께 보기 쉽게 출력한다."""
    if not result:
        print("분석 결과가 없습니다. 체인 실행 여부를 먼저 확인하세요.")
        return

    if isinstance(result, dict) and "indicator_json" in result:
        print("[1. 입력된 JSON 지표]")
        print("=" * 60)
        print(result["indicator_json"])
        print("=" * 60 + "\n")

    categories = [
        ("pattern_result", "[2. 소비 패턴 분석]"),
        ("problem_result", "[3. 문제 소비 식별]"),
        ("cause_result", "[4. 소비 원인 해석]"),
        ("action_result", "[5. 행동 개선 제안]"),
    ]

    if isinstance(result, dict):
        for key, title in categories:
            if key not in result:
                continue
            print(title)
            print("-" * 60)
            data = result[key]
            if hasattr(data, "model_dump_json"):
                print(data.model_dump_json(indent=2))
            else:
                print(data)
            print("-" * 60 + "\n")


print_pretty_json_analysis(analysis_result)

[1. 입력된 JSON 지표]
{
  "member_id": 1,
  "analysis_date": "2024-04-01",
  "metrics": [
    {
      "name": "과거 안정 일평균",
      "value": 51014.0549,
      "unit": "KRW",
      "source_json_path": "stable_metrics.past_daily_stable_average",
      "description": "클리핑 데이터 기준 과거 일평균 소비 금액"
    },
    {
      "name": "오늘 총 지출액",
      "value": 133044,
      "unit": "KRW",
      "source_json_path": "stable_metrics.today_total",
      "description": "분석 기준일의 총 소비 금액"
    },
    {
      "name": "평소 대비 지출 증가율",
      "value": 160.7987,
      "unit": "percent",
      "source_json_path": "stable_metrics.increase_rate_percent",
      "description": "과거 안정 일평균 대비 오늘 총 지출 증가율"
    },
    {
      "name": "원본 일평균 대비 소비 배율",
      "value": 1.2741,
      "unit": "ratio",
      "source_json_path": "anomaly_detection.spike_ratio",
      "description": "원본 데이터 과거 일평균 대비 오늘 지출 배율"
    },
    {
      "name": "소비 급증 여부",
      "value": false,
      "unit": "boolean",
      "source_json_path": "anomaly_detection.i